# Day 2 · Lab 1 — Multi-Agent Loan Processing (Supervisor Pattern)

## What you'll build

A supervisor-pattern multi-agent system where 3 specialist agents share typed state:

1. **Supervisor**: routes work to specialists based on `current_agent` field
2. **Eligibility Agent**: deterministic income/LTV check
3. **Bureau Agent**: LLM-simulated credit score lookup
4. **Memo Agent**: LLM-generated credit memo

All 4 nodes share a single `MultiAgentLoanState` via LangGraph's typed state model. Field ownership is enforced by convention (one node writes each field).

## Prerequisites

- Day 1 Lab 1 completed and working
- Same sandbox: `~/agentic-lab/.env` has `OPENROUTER_API_KEY`, `DATABASE_URL`, Postgres up
- Same kernel: `/opt/miniconda/bin/python` (base)

## Step 1 — Environment diagnostic

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"])
    from dotenv import load_dotenv

env_path = Path("~/agentic-lab/.env").expanduser()
load_dotenv(env_path, override=False)
print(f"✓ .env loaded from {env_path}")

for k in list(os.environ.keys()):
    if k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY","OPENROUTER_API_KEY","DATABASE_URL") and os.environ.get(k) == "":
        del os.environ[k]

required = {
    "OPENROUTER_API_KEY": "LLM gateway",
    "DATABASE_URL":       "PostgreSQL for shared checkpoints",
}
all_ok = True
for k, desc in required.items():
    v = os.environ.get(k)
    print(f"  {'✓' if v else '✗'} {k}  ({desc})")
    if not v: all_ok = False

if os.environ.get("LANGSMITH_TRACING","").lower() == "true" and not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("  → LANGSMITH_TRACING flipped to false (no key)")

if os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

print("\n" + ("✓ Continue." if all_ok else "⚠  Fix ✗ above."))

## Step 2 — Multi-agent state schema

The KEY design decision: which fields does each agent own?

- `eligibility_agent` writes `eligibility_result`
- `bureau_agent` writes `bureau_result`
- `memo_agent` writes `memo_draft`
- `agent_log` uses `operator.add` — every agent APPENDS its trace entries
- `current_agent` + `completed` are supervisor routing signals

In [ ]:
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
import operator


class EligibilityResult(BaseModel):
    passed: bool
    score: float = Field(ge=0, le=100)
    reason: str


class BureauResult(BaseModel):
    score: int = Field(ge=300, le=850)
    tier: Literal["low", "medium", "high"]


class MultiAgentLoanState(TypedDict):
    application_id: str
    monthly_income: float
    loan_amount: float

    # Owned fields — ONE agent writes each
    eligibility_result: dict
    bureau_result: dict
    memo_draft: str

    # Accumulative
    agent_log: Annotated[list, operator.add]

    # Supervisor routing signals
    current_agent: str
    completed: Annotated[list, operator.add]


print("Multi-agent state schema defined.")

## Step 3 — LLM setup (OpenRouter)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)
print("LLM configured for OpenRouter.")

## Step 4 — Eligibility Agent (deterministic)

In [ ]:
def eligibility_agent(state: MultiAgentLoanState) -> dict:
    income = state["monthly_income"]
    amount = state["loan_amount"]
    ratio = income * 12 / amount if amount > 0 else 0
    score = min(100.0, ratio * 25)
    passed = score >= 50

    result = EligibilityResult(
        passed=passed,
        score=score,
        reason=f"annual income / loan = {ratio:.2f}"
    ).model_dump()

    return {
        "eligibility_result": result,
        "agent_log": [f"eligibility_agent: score={score:.1f} passed={passed}"],
        "completed": ["eligibility"],
    }


sample = {"monthly_income": 6000, "loan_amount": 200_000, "agent_log": [], "completed": []}
print(eligibility_agent(sample))

## Step 5 — Bureau Agent (LLM-backed)

In [ ]:
import json


def bureau_agent(state: MultiAgentLoanState) -> dict:
    prompt = (
        f"Applicant with monthly income {state['monthly_income']:.0f}, "
        f"loan amount {state['loan_amount']:.0f}. Return a plausible credit "
        f"bureau JSON with keys 'score' (int 300-850) and 'tier' (low/medium/high). "
        f"ONLY JSON, no prose."
    )
    text = llm.invoke(prompt).content.strip()
    if text.startswith("```"):
        text = text.strip("`").split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        if text.startswith("json"):
            text = text[4:].strip()
    try:
        data = json.loads(text)
        result = BureauResult(**data).model_dump()
    except Exception:
        result = BureauResult(score=700, tier="medium").model_dump()

    return {
        "bureau_result": result,
        "agent_log": [f"bureau_agent: {result}"],
        "completed": ["bureau"],
    }


print("Bureau agent defined.")

## Step 6 — Memo Agent (LLM-backed)

In [ ]:
def memo_agent(state: MultiAgentLoanState) -> dict:
    elig = state.get("eligibility_result", {})
    bureau = state.get("bureau_result", {})
    prompt = (
        f"Draft a 2-sentence credit memo for application {state['application_id']}. "
        f"Loan {state['loan_amount']:.0f}, monthly income {state['monthly_income']:.0f}. "
        f"Eligibility: {elig}. Bureau: {bureau}. Be factual, no fluff."
    )
    draft = llm.invoke(prompt).content.strip()

    return {
        "memo_draft": draft,
        "agent_log": [f"memo_agent: drafted {len(draft)} chars"],
        "completed": ["memo"],
    }


print("Memo agent defined.")

## Step 7 — Supervisor (routing)

Reads `completed`, picks the next specialist. Short-circuits to END on eligibility failure.

In [ ]:
def supervisor(state: MultiAgentLoanState) -> dict:
    done = set(state.get("completed", []))

    if "eligibility" not in done:
        next_agent = "eligibility"
    elif not state.get("eligibility_result", {}).get("passed", False):
        next_agent = "end"
    elif "bureau" not in done:
        next_agent = "bureau"
    elif "memo" not in done:
        next_agent = "memo"
    else:
        next_agent = "end"

    return {
        "current_agent": next_agent,
        "agent_log": [f"supervisor: routing to {next_agent}"],
    }


def route_from_supervisor(state: MultiAgentLoanState) -> str:
    return state["current_agent"]


print("Supervisor defined.")

## Step 8 — Assemble the graph

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(MultiAgentLoanState)
builder.add_node("supervisor", supervisor)
builder.add_node("eligibility", eligibility_agent)
builder.add_node("bureau", bureau_agent)
builder.add_node("memo", memo_agent)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"eligibility": "eligibility", "bureau": "bureau", "memo": "memo", "end": END},
)
builder.add_edge("eligibility", "supervisor")
builder.add_edge("bureau", "supervisor")
builder.add_edge("memo", "supervisor")

print("Graph assembled.")

## Step 9 — Compile with PostgresSaver

In [ ]:
from langgraph.checkpoint.postgres import PostgresSaver

_pg_ctx = PostgresSaver.from_conn_string(os.environ["DATABASE_URL"])
checkpointer = _pg_ctx.__enter__()
checkpointer.setup()

graph = builder.compile(checkpointer=checkpointer)
print("Graph compiled with PostgresSaver.")

## Step 10 — Run end-to-end

In [ ]:
import uuid

app_id = f"MULTI-{uuid.uuid4().hex[:8].upper()}"
config = {"configurable": {"thread_id": f"multi-{app_id}"}}

initial_state = {
    "application_id": app_id,
    "monthly_income": 6000,
    "loan_amount": 200_000,
    "eligibility_result": {},
    "bureau_result": {},
    "memo_draft": "",
    "agent_log": [],
    "current_agent": "",
    "completed": [],
}

print(f"Running multi-agent workflow for {app_id}...\n")
result = graph.invoke(initial_state, config, {"recursion_limit": 20})

print("─" * 60)
print("Agent log:")
for entry in result["agent_log"]:
    print(" ", entry)

print("\nFinal state:")
print(f"  eligibility: {result['eligibility_result']}")
print(f"  bureau:      {result['bureau_result']}")
print(f"  memo:        {result['memo_draft'][:120]}...")
print(f"  completed:   {result['completed']}")

## Step 11 — Failing case

Low income = eligibility fails = supervisor short-circuits, bureau and memo never run.

In [ ]:
fail_id = f"FAIL-{uuid.uuid4().hex[:8].upper()}"
fail_config = {"configurable": {"thread_id": f"multi-{fail_id}"}}

fail_state = {
    "application_id": fail_id,
    "monthly_income": 1500,
    "loan_amount": 200_000,
    "eligibility_result": {},
    "bureau_result": {},
    "memo_draft": "",
    "agent_log": [],
    "current_agent": "",
    "completed": [],
}

result = graph.invoke(fail_state, fail_config, {"recursion_limit": 20})
print("Agent log (failing case):")
for entry in result["agent_log"]:
    print(" ", entry)
print(f"\nBureau ran? {'bureau' in result['completed']}")
print(f"Memo ran?   {'memo' in result['completed']}")

## Cleanup

In [ ]:
_pg_ctx.__exit__(None, None, None)
print("PostgresSaver closed.")

## What you learned

1. **Supervisor pattern** — one node decides routing; specialists never call each other
2. **Field ownership** — one agent per field; enforced by convention
3. **Accumulative fields** — `agent_log` with `operator.add` gives a full trace
4. **Routing signals** — `current_agent` + `completed` drive the loop
5. **Short-circuit logic** — supervisor can jump to END based on state
6. **Pydantic contracts** — `EligibilityResult`/`BureauResult` validate agent outputs

## Next

Open `lab2_fastmcp_server.ipynb` to build a custom MCP server with auth + rate limiting.
